# Example 06 — Multi-DOF, Multi-Nonlinearity FRF

FRF for all 3 DOFs of a 3-DOF chain with multiple nonlinear elements:

- `elastic_dry_friction` (Jenkins/Masing element, k_slip=20, f_lim=1) at DOF 0 and relative DOF 0–1
- `cubic_spring` (k3=1) at DOF 1 and relative DOF 1–2
- `unilateral_spring` (k=1, gap=0.25) at DOF 2

Parameters match the MATLAB/Octave NLvib reference (Krack & Gross 2019, Example 07).


In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.systems.oscillators import ChainOfOscillators
from nlvib.nonlinearities.elements import (
    cubic_spring, elastic_dry_friction, polynomial_stiffness, unilateral_spring
)
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions


In [ ]:
# --- System setup (matches MATLAB NLvib Example 07) ---
# 3-DOF chain: mi=[1,1,1], ki=[1,1,1,1], di=0.02*ki, Fex1=[0;1;0]
MASSES      = [1.0, 1.0, 1.0]
STIFFNESSES = [1.0, 1.0, 1.0, 1.0]
DAMPINGS    = [0.02, 0.02, 0.02, 0.02]
FORCE_AMP   = 1.0
OMEGA_MIN   = 0.5
OMEGA_MAX   = 2.0
N_HARMONICS = 3   # H=3 sufficient for Jenkins/cubic/unilateral

# Jenkins element parameters (MATLAB: knl=20, muN=1)
K_SLIP = 20.0
F_LIM  = 1.0
W1 = np.array([1.0, 0.0, 0.0])   # DOF 0 ground
W2 = np.array([-1.0, 1.0, 0.0])  # relative DOF 0-1

system = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)

# 1. Jenkins element on DOF 0 (W1=[1,0,0])
system.add_nonlinear_element(elastic_dry_friction(k_slip=K_SLIP, f_lim=F_LIM, dof_index=0))
# 2. Jenkins element on relative DOF 0-1 (W2=[-1,1,0])
system.add_nonlinear_element(elastic_dry_friction(k_slip=K_SLIP, f_lim=F_LIM, force_direction=W2))
# 3. cubic_spring W3=[0;1;0] -> DOF 1, k3=1
system.add_nonlinear_element(cubic_spring(k3=1.0, dof_index=1))
# 4. cubic_spring W4=[0;-1;1] -> relative DOF 1-2 (polynomial trick)
_k3_rel    = 1.0
_exp_rel   = np.array([[3, 0], [2, 1], [1, 2], [0, 3]], dtype=np.intp)
_coeff_rel = np.array([_k3_rel, -3*_k3_rel, 3*_k3_rel, -_k3_rel])
system.add_nonlinear_element(polynomial_stiffness(_exp_rel, _coeff_rel, np.array([1, 2], dtype=np.intp)))
system.add_nonlinear_element(polynomial_stiffness(_exp_rel, _coeff_rel, np.array([2, 1], dtype=np.intp)))
# 5. unilateral_spring W5=[0;0;1] -> DOF 2, k=1, gap=0.25
system.add_nonlinear_element(unilateral_spring(k=1.0, gap=0.25, dof_index=2))

excitation = {'dof': 1, 'amplitude': FORCE_AMP, 'harmonic': 1}
n_dof   = system.n_dof
H       = N_HARMONICS
n_total = n_dof * (2*H + 1)
print(f'System: n_dof={n_dof}, H={H}, n_total={n_total}')
print(f'Nonlinear elements: {len(system.nonlinear_elements)}')


In [ ]:
# --- Initial solution using Jenkins stuck-stiffness linear guess ---
# MATLAB: dKfric = knl*(W1*W1' + W2*W2'); Q1 = (-Om^2*M + i*Om*D + K + dKfric)\Fex1
dKfric = K_SLIP * (np.outer(W1, W1) + np.outer(W2, W2))
K_eff  = system.K.toarray() + dKfric
M_arr  = system.M.toarray()
D_arr  = system.D.toarray()
Fex1   = np.array([0.0, FORCE_AMP, 0.0])

omega0 = OMEGA_MIN
Q1_c = np.linalg.solve(-(omega0**2)*M_arr + 1j*omega0*D_arr + K_eff, Fex1)
Q0 = np.zeros(n_total, dtype=np.float64)
for dof_idx in range(n_dof):
    Q0[n_dof*1 + dof_idx] =  float(np.real(Q1_c[dof_idx]))   # cos1
    Q0[n_dof*2 + dof_idx] = -float(np.imag(Q1_c[dof_idx]))   # sin1

for _it in range(50):
    R, J = hb_residual(Q0, omega0, system, H, excitation)
    if np.linalg.norm(R) < 1e-10:
        break
    try:
        dQ = np.linalg.solve(J, -R)
    except np.linalg.LinAlgError:
        dQ = np.linalg.lstsq(J, -R, rcond=None)[0]
    Q0 = Q0 + dQ

print(f'Initial residual at omega={omega0:.3f}: {np.linalg.norm(R):.3e}')

def residual_fn(Q, omega):
    return hb_residual(Q, omega, system, H, excitation)

opts = ContinuationOptions(
    verbose=False,
    ds_initial=0.01,
    ds_min=1e-6,
    ds_max=0.05,
    max_steps=300,
    newton_tol=1e-8,
    lambda_min=OMEGA_MIN,
    lambda_max=OMEGA_MAX,
)
solver = ContinuationSolver()
result = solver.run(residual_fn, Q0, omega0, opts)
print(f'Continuation: {result.n_steps} steps, {result.message}')


In [ ]:
# --- Extract amplitudes (a_rms per DOF) and save plots ---
solutions  = result.solutions           # shape: (n_steps, n_total + 1)
stability  = result.stability
omegas     = solutions[:, -1]
Q_all      = solutions[:, :-1]          # shape: (n_steps, n_dof*(2H+1))

# Reshape: (n_steps, 2H+1, n_dof)
Q_reshaped = Q_all.reshape(len(omegas), 2*H + 1, n_dof)

# a_rms per DOF — matches MATLAB formula: sqrt(sum(Q.^2)) / sqrt(2)
a_rms = np.zeros((len(omegas), n_dof))
for dof in range(n_dof):
    Q_dof = Q_reshaped[:, :, dof]
    a_rms[:, dof] = np.sqrt(np.sum(Q_dof**2, axis=1)) / np.sqrt(2)

output_dir = Path('..') / 'examples' / '06_multi_dof_multi_nl' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

dof_colors = ['steelblue', 'darkorange', 'forestgreen']
dof_labels = ['DOF 0 (Jenkins grnd)', 'DOF 1 (cubic spring)', 'DOF 2 (unilateral)']

fig_frf, axes_frf = plt.subplots(1, n_dof, figsize=(15, 5), sharey=False)
for dof, (ax, color, lbl) in enumerate(zip(axes_frf, dof_colors, dof_labels)):
    for i in range(len(omegas) - 1):
        is_stable = not bool(stability[i])
        ax.plot(omegas[i:i+2], a_rms[i:i+2, dof],
                color=color if is_stable else 'tab:red',
                linestyle='-' if is_stable else '--', linewidth=1.5)
    ax.set_xlabel(r'Excitation frequency $\Omega$ (rad/s)')
    ax.set_ylabel('Response amplitude RMS')
    ax.set_title(f'Example 06 — {lbl}')
    ax.set_xlim(OMEGA_MIN, OMEGA_MAX)
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5)

handles = [
    Line2D([0], [0], color='gray', linestyle='-', label='stable'),
    Line2D([0], [0], color='tab:red', linestyle='--', label='unstable'),
]
axes_frf[0].legend(handles=handles, loc='upper left', fontsize=8)
fig_frf.suptitle('Example 06 — Multi-DOF Multi-NL FRF (elastic_dry_friction + cubic + unilateral)')
fig_frf.tight_layout()
fig_frf.savefig(output_dir / 'frf_all_dofs.png', dpi=150)
print('Saved frf_all_dofs.png')

fig_conv, ax_conv = plt.subplots(figsize=(8, 4))
ds_history = result.ds_history[1:]
ax_conv.semilogy(np.arange(1, len(ds_history)+1), ds_history, color='tab:green', linewidth=1.2)
ax_conv.set_xlabel('Continuation step')
ax_conv.set_ylabel('Arc-length step size ds')
ax_conv.set_title('Example 06 — Newton Convergence Proxy (ds history)')
ax_conv.grid(True, which='both', alpha=0.4)
fig_conv.tight_layout()
fig_conv.savefig(output_dir / 'convergence.png', dpi=150)
print('Saved convergence.png')


In [ ]:
# --- Display FRF (all DOFs) inline ---
fig_frf


In [ ]:
# --- Display convergence proxy inline ---
fig_conv


In [ ]:
# --- Summary ---
print('=' * 60)
print('SUMMARY — Example 06: Multi-DOF Multi-NL Peak Amplitudes')
print('  (elastic_dry_friction k_slip=20 f_lim=1; cubic k3=1; unilateral k=1 gap=0.25)')
print('=' * 60)
for dof in range(n_dof):
    amp = a_rms[:, dof]
    if len(amp) > 0:
        peak_idx = int(np.argmax(amp))
        print(f'  DOF {dof}: peak a_rms = {float(amp[peak_idx]):.6f}  at omega = {float(omegas[peak_idx]):.4f} rad/s')
    else:
        print(f'  DOF {dof}: no data')
print('=' * 60)
